In [1]:
import os
from datetime import datetime
from sys import exit
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from re import X

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F

os.makedirs("models", exist_ok=True)
os.makedirs("ensemble_models", exist_ok=True)

def set_all_seeds(seed):
    np.random.seed(seed)
    print(f"Random Seed: {seed}")
    random.seed(seed)
    torch.manual_seed(seed)

seed = 0
set_all_seeds(seed)



Random Seed: 0


In [2]:

df = pd.read_csv(f"BTCUSDT-15m-data.csv")
print(df.head())

             timestamp      open_time     open     high      low    close  \
0  2017-08-17 04:00:00  1502942400000  4261.48  4280.56  4261.48  4261.48   
1  2017-08-17 04:15:00  1502943300000  4261.48  4270.41  4261.32  4261.45   
2  2017-08-17 04:30:00  1502944200000  4280.00  4310.07  4267.99  4310.07   
3  2017-08-17 04:45:00  1502945100000  4310.07  4313.62  4291.37  4308.83   
4  2017-08-17 05:00:00  1502946000000  4308.83  4328.69  4304.31  4304.31   

      volume     close_time      quote_av  trades  tb_base_av   tb_quote_av  \
0   2.189061  1502943299999   9333.620962       9    0.489061   2089.104962   
1   9.119865  1502944199999  38891.133046      40    3.447113  14703.934995   
2  21.923552  1502945099999  94080.917568      58   20.421317  87620.977876   
3  13.948531  1502945999999  60060.466816      64   10.803012  46538.460109   
4   5.101153  1502946899999  22006.533111      44    3.496635  15093.783057   

   ignore  
0       0  
1       0  
2       0  
3       0  
4 

In [3]:
df.count()

,0
timestamp,276000
open_time,276000
open,276000
high,276000
low,276000
close,276000
volume,276000
close_time,276000
quote_av,276000
trades,276000


In [4]:
df.tail()

,timestamp,open_time,open,high,low,close,volume,close_time,quote_av,trades,tb_base_av,tb_quote_av,ignore
275995,2025-07-07 00:00:00,1751846400000,109203.85,109273.07,109103.19,109273.06,56.48447,1751847299999,6.166301e+06,26415,26.22332,2.862917e+06,0
275996,2025-07-07 00:15:00,1751847300000,109273.07,109288.02,108972.69,109004.81,75.36437,1751848199999,8.220317e+06,24191,33.56157,3.660548e+06,0
275997,2025-07-07 00:30:00,1751848200000,109004.81,109037.82,108829.17,108847.16,76.72080,1751849099999,8.357724e+06,21368,33.08997,3.605029e+06,0
275998,2025-07-07 00:45:00,1751849100000,108847.17,108922.00,108800.01,108823.07,45.06548,1751849999999,4.906468e+06,15579,22.76836,2.478881e+06,0
275999,2025-07-07 01:00:00,1751850000000,108823.07,108823.08,108679.75,108767.01,75.71845,1751850899999,8.232864e+06,22928,38.85713,4.224608e+06,0


In [5]:
print(df.isnull())
print(f'Counts how many missing values there are in each column: {df.isnull().sum()}')
print(f'Total missing values: {df.isnull().sum().sum()}')

        timestamp  open_time   open   high    low  close  volume  close_time  \
0           False      False  False  False  False  False   False       False   
1           False      False  False  False  False  False   False       False   
2           False      False  False  False  False  False   False       False   
3           False      False  False  False  False  False   False       False   
4           False      False  False  False  False  False   False       False   
...           ...        ...    ...    ...    ...    ...     ...         ...   
275995      False      False  False  False  False  False   False       False   
275996      False      False  False  False  False  False   False       False   
275997      False      False  False  False  False  False   False       False   
275998      False      False  False  False  False  False   False       False   
275999      False      False  False  False  False  False   False       False   

        quote_av  trades  tb_base_av  t

#Training Phase

Splitting data to train/test sets

Practice same pattersn multiple times

Have access to answer key

problems known:
Overfitting
Underfitting

#Testing Phase

Face new Patterns

No access to asnwer key


In [6]:
df[['open', 'high', 'low', 'close', 'volume']].head()

,open,high,low,close,volume
0,4261.48,4280.56,4261.48,4261.48,2.189061
1,4261.48,4270.41,4261.32,4261.45,9.119865
2,4280.00,4310.07,4267.99,4310.07,21.923552
3,4310.07,4313.62,4291.37,4308.83,13.948531
4,4308.83,4328.69,4304.31,4304.31,5.101153


In [7]:

df['open'].dtypes

dtype('float64')

In [8]:
df['open_time'].diff().dropna().value_counts().head()

,count
open_time,
900000.0,275966
4500000.0,5
8100000.0,3
1800000.0,2
7200000.0,2


In [9]:
# what is a feature:
# a Mesurable piece of information that describe the data
# help the model to recognize patterns

#always pay attention to the shape of the data

def compute_moving_average(prices, window):
    values = np.full(len(prices), np.nan, dtype=np.float64)
    for i in range(window - 1, len(prices)):
        values[i] = prices[i - window + 1:i + 1].mean()  # FIX: mean, not max
    return values

def compute_moving_std(prices, window):
    values = np.full(len(prices), np.nan, dtype=np.float64)
    for i in range(window - 1, len(prices)):
        values[i] = prices[i - window + 1:i + 1].std()
    return values

def build_feature(opens, highs, lows, closes, volumes, train_scalers=None):

    #EPSILON
    eps = 1e-12

    #build features
    #candle shape features (normalized)
    feature1 = (closes - opens) / (opens + eps)
    feature2 = (highs - opens) / (opens + eps) # how much price rose above the open
    feature3 = (highs - closes) / (closes + eps) # how far the close was below the high
    feature4 = (lows - opens) / (opens + eps) # how much price fell below the open

    # “How far the low was below the close” (lower wick strength)
    feature5 = (closes - lows) / (closes + eps)

    feature6 = (highs - lows) / (opens + eps) # the day's volatility range relative to open
    feature7 = (highs - lows) / (closes + eps) # the day's volatility range relative to close

    # Volume: heavy-tailed -> stabilize
    feature8 = np.log1p(volumes)

    # MA deviation + volatility
    close_ma_5 = compute_moving_average(closes, 5)
    feature9 = (closes - close_ma_5) / (close_ma_5 + eps)

    feature10 = compute_moving_std(closes, 5)

    # Stack features
    features = [
      feature1, feature2,
      feature3, feature4,
      feature5, feature6,
      feature7, feature8,
      feature9, feature10
    ]

    # Drop rows with NaN (from rolling features)
    feat_mat = np.stack(features, axis=-1)  # (m, 10)
    valid_mask = ~np.isnan(feat_mat).any(axis=1)

    #feature matrix shape(m, 10) 10 = n features
    feat_mat = feat_mat[valid_mask]

    num_features = feat_mat.shape[1]

    scaled_fts = []

    if train_scalers is None:
      train_scalers = [MaxAbsScaler() for _ in range(num_features)]
      is_train = True
    else:
      is_train = False

    scaled_cols = []
    for i in range(num_features):
        col = feat_mat[:, i].reshape(-1, 1)
        scaler = train_scalers[i]
        if is_train:
            col_s = scaler.fit_transform(col)
        else:
            col_s = scaler.transform(col)
        scaled_cols.append(col_s.reshape(-1))

    scaled_features = np.stack(scaled_cols, axis=-1)  # (m_valid, 10)
    return scaled_features, num_features, train_scalers, valid_mask

In [10]:
# # This list will store each feature AFTER scaling
# # Each element will be a 1D array (one column / one feature)
# scaled_cols = []

# # Loop through each feature (column-wise scaling)
# for i in range(num_features):

#     # Extract ONE feature column from the feature matrix
#     # feat_mat shape: (num_rows, num_features)
#     # col shape becomes: (num_rows, 1)
#     # We reshape because sklearn scalers expect 2D input
#     col = feat_mat[:, i].reshape(-1, 1)

#     # Get the scaler corresponding to this feature
#     # Each feature has its own scaler to avoid mixing distributions
#     scaler = train_scalers[i]

#     if is_train:
#         # TRAINING MODE
#         # fit_transform():
#         # 1) LEARNS the scaling parameters (e.g. max value)
#         # 2) APPLIES the scaling to the training data
#         # This MUST happen ONLY on training data to avoid data leakage
#         col_s = scaler.fit_transform(col)

#     else:
#         # TESTING / INFERENCE MODE
#         # transform():
#         # Uses the SAME parameters learned from training
#         # IMPORTANT: We do NOT re-learn anything here
#         # This simulates real live trading where future data is unseen
#         col_s = scaler.transform(col)

#     # Flatten back to 1D so we can stack columns later
#     scaled_cols.append(col_s.reshape(-1))

# # Stack all scaled feature columns back together
# # Resulting shape: (num_rows, num_features)
# # Each row = one candle
# # Each column = one scaled feature
# scaled_features = np.stack(scaled_cols, axis=-1)

# # Return:
# # scaled_features → clean, scaled input for the model
# # num_features    → number of input features (used by the model)
# # train_scalers   → saved scalers (needed for test/live data)
# # valid_mask      → tells which original rows were kept (after NaN removal)
# return scaled_features, num_features, train_scalers, valid_mask



Summary table (easy recall)
Choice	Why
to_numpy()	Clean pandas → NumPy conversion
float64 (features)	Accurate math, stable rolling stats
float32 (X, Y)	Fast ML training, GPU friendly
m = len(features)	Only valid rows count
samples	Sliding windows, not candles
num_samples	Prevent leakage & index errors


One-sentence mental model
Clean and compute precisely first (float64), then train efficiently (float32), always counting only valid data.

In [10]:
def preprocessing_data(seq_len, df, train_scalers=None, horizon=4):

    # seq_len=96: input window (24h for 15m)
    # horizon=4: predict 1-hour forward return (4*15m)

    opens   = df['open'].to_numpy(dtype=np.float64)
    highs   = df['high'].to_numpy(dtype=np.float64)
    lows    = df['low'].to_numpy(dtype=np.float64)
    closes  = df['close'].to_numpy(dtype=np.float64)
    volumes = df['volume'].to_numpy(dtype=np.float64)

    # Build features
    features, num_features, train_scalers, valid_mask = build_feature(opens, highs, lows, closes, volumes, train_scalers)

    # Align opens/closes with the same valid_mask used for features
    opens_valid = opens[valid_mask]
    closes_valid = closes[valid_mask]


    #Number of valid time steps after feature construction.
    m = len(features)
    num_samples = m - seq_len - horizon
    if num_samples <= 0:
        raise ValueError("Not enough data after dropping NaNs for chosen seq_len/horizon.")

    #create storage for input & targets
    # x the input sequence
    # y the output/target
    X = np.zeros((num_samples, seq_len, num_features), dtype=np.float32)
    Y = np.zeros((num_samples, 1), dtype=np.float32)  # FIX: regression target is 1 value
    opens_bt  = np.zeros(num_samples, dtype=np.float64)
    closes_bt = np.zeros(num_samples, dtype=np.float64)

    for i in range(num_samples):
        X[i] = features[i:i+seq_len]

        # FUTURE LOG RETURN target at the prediction point
        t = i + seq_len
        opens_bt[i]  = opens_valid[t] #price at decision time (when model acts)
        closes_bt[i] = closes_valid[t + horizon] #price after holding for horizon candles

        y = np.log(closes_valid[t + horizon] / closes_valid[t])
        Y[i, 0] = y

    return X, Y, opens_bt, closes_bt, num_features, train_scalers

Return prices at the same timestamps your model predicts on — never raw dataframe prices.

In [11]:
# Output matches regression target (B,1)

#  Avoids flattening (64*96) which overfits

#  Residual improves training stability

#  Fast, strong baseline for candles


class Model(nn.Module):
    def __init__(self, num_features, seq_len, hidden=64, dropout=0.1):
        super().__init__()

        # Project features -> hidden channels
        self.in_proj = nn.Conv1d(num_features, hidden, kernel_size=1)

        # Two temporal conv blocks (can add more)
        self.conv1 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)

        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

        # Regression head: use LAST timestep representation (causal-ish)
        self.fc_out = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (B, T, F)
        x = x.transpose(1, 2)          # (B, F, T)

        x = self.in_proj(x)            # (B, H, T)

        # Conv block
        y = self.act(self.conv1(x))
        y = self.drop(y)
        y = self.act(self.conv2(y))
        y = self.drop(y)

        # Residual connection helps stability
        x = x + y                      # (B, H, T)

        # Take last timestep representation
        last = x[:, :, -1]             # (B, H)

        #Regression head Converts the learned representation into: predicted future return
        #Exactly matches Y_train.shape = (N, 1)
        out = self.fc_out(last)        # (B, 1)
        return out

Conceptually, it’s just:
[96 candles]
     ↓
extract time patterns
     ↓
summarize into one vector
     ↓
predict future return

The forward function turns 96 past candles into a learned summary, then maps that summary to a future return estimate.

In [23]:
def create_minibatches(X, Y, batch_size, seed, shuffle=True):
    #mini_batches will hold a list of (mb_X, mb_Y) pairs.
    mini_batches = []

    #m = number of training samples (windows).
    m = len(X)

    #Creates a random order of indices [0..m-1].
    #Reorders X and Y the same way, so inputs still match targets.
    #Why: helps SGD/Adam generalize and not learn “the order”.

    if shuffle:
        # RNG Numpy Random Generator
        rng = np.random.default_rng(seed) # seed controls shuffle
        permutation = rng.permutation(m)
        X = X[permutation]
        Y = Y[permutation]
    #Compute how many full batches fit
    num_batches = m // batch_size
    #Slice batches
    for k in range(num_batches):
        start = k * batch_size
        end = start + batch_size
        mini_batches.append((X[start:end], Y[start:end]))


    #Seed update
    # seed += 1
    set_all_seeds(seed)

    return mini_batches, seed

One-line rule to remember

If you index NumPy arrays, keep indices as NumPy arrays.

So: do not convert perm to list()

In [24]:
from re import X
# Sequence Lenght fot the inout sequence equivalent 1 day of 15m candles
seq_len = 96
horizon = 4  # 1 hour ahead for 15m candles
num_epochs = 100

In [25]:
#Train one epoch
def train_one_epoch(model, optimizer, criterion, mini_batches):
    model.train()
    batch_losses = []

    for Xb, Yb in mini_batches:
        optimizer.zero_grad()
        pred = model(Xb)                     # (B, 1)
        loss = criterion(pred, Yb)           # scalar
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    return float(np.mean(batch_losses))

model.eval() switches the network into inference mode, turning off training-only behavior (like dropout) so validation/test results are stable and meaningful.

In [26]:
#Evaluate (test loss + predictions)

@torch.no_grad()
def eval_model(model, criterion, X_test, Y_test):
    model.eval()
    pred = model(X_test)                     # (N, 1)
    loss = criterion(pred, Y_test).item()
    return loss, pred

In [27]:
#Turn regression predictions into signals (thresholded)
def predictions_to_signals(pred_return, thr):
    """
    pred_return: numpy (N,1) predicted log-return
    thr: threshold in log-return units
    returns: signals in {-1,0,+1}
    """
    p = pred_return[:, 0]
    sig = np.zeros_like(p, dtype=np.int8)
    sig[p >  thr] =  1   # long
    sig[p < -thr] = -1   # short
    return sig

In [28]:
#Backtest (aligned with our horizon strategy)
#This assumes you already return opens_bt and closes_bt from preprocessing (aligned to samples).

def backtest_signals(opens_bt, closes_bt, signals, pos_size=1000.0, fee_rate=0.0):
    """
    opens_bt[i]  = entry reference price at time t
    closes_bt[i] = exit reference price at time t+horizon
    signals[i] in {-1,0,+1}
    pos_size in USD notionals per trade

    fee_rate example:
      - If you want 0.04% taker each side: fee_rate = 0.0004
      - Round trip fees approx: 2 * fee_rate * pos_size per trade (rough)
    """
    equity = 1000.0
    equities = []
    correct = 0
    traded = 0
    n = len(signals)

    i = 0
    while i < n:
        s = signals[i]
        if s == 0:
            equities.append(equity)
            i += 1
            continue

        o = opens_bt[i]
        c = closes_bt[i]  # already aligned to t+horizon in your preprocessing
        pct_change = (c - o) / o

        pnl = s * pos_size * pct_change
        pnl -= 2.0 * fee_rate * pos_size

        equity += pnl
        equities.append(equity)

        traded += 1
        if (s == 1 and pct_change > 0) or (s == -1 and pct_change < 0):
            correct += 1

        i += horizon  # IMPORTANT: skip forward so trades don’t overlap

    accuracy = (correct / traded * 100.0) if traded > 0 else 0.0
    return equity, np.array(equities, dtype=np.float64), accuracy, traded

In [29]:
#Save checkpoint (use a new filename for new architecture)
def save_checkpoint(path, epoch, model, optimizer, train_losses, test_losses, equity_epochs, accuracy_epochs, seed):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "test_losses": test_losses,
        "equity_epochs": equity_epochs,
        "accuracy_epochs": accuracy_epochs,
        "seed": seed,
    }, path)

In [30]:
def choose_threshold_from_val(pred_val_np, q=0.90):
    # trade top 10% strongest signals
    p = pred_val_np[:,0]
    thr = np.quantile(np.abs(p), q)
    return float(thr)


In [31]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("my device: ", device)

my device:  cuda


In [32]:
fee_rate = 0.0005 # example placeholder (0.04%); set to 0 if you ignore fees
wf_results = []

start = 0
fold = 0
batch_size = 64

CANDLES_PER_DAY = 96
TRAIN = 365 * CANDLES_PER_DAY   # 12 months
VAL   =  90 * CANDLES_PER_DAY   # 3 months
TEST  =  90 * CANDLES_PER_DAY   # 3 months
STEP  =  TEST                   # roll 3 months

while start + TRAIN + VAL + TEST <= len(df):

    df_train = df.iloc[start : start + TRAIN].copy()
    df_val   = df.iloc[start + TRAIN : start + TRAIN + VAL].copy()
    df_test  = df.iloc[start + TRAIN + VAL : start + TRAIN + VAL + TEST].copy()

    # ---- preprocess (fit scalers on TRAIN only) ----
    X_train, Y_train, opens_tr, closes_tr, num_features, scalers = preprocessing_data(
        seq_len, df_train, train_scalers=None, horizon=horizon
    )
    X_val, Y_val, opens_val, closes_val, _, _ = preprocessing_data(
        seq_len, df_val, train_scalers=scalers, horizon=horizon
    )
    X_test, Y_test, opens_test, closes_test, _, _ = preprocessing_data(
        seq_len, df_test, train_scalers=scalers, horizon=horizon
    )

    # ---- torch tensors ----
    X_train_t = torch.from_numpy(X_train).to(device)
    Y_train_t = torch.from_numpy(Y_train).to(device)

    X_val_t   = torch.from_numpy(X_val).to(device)
    Y_val_t   = torch.from_numpy(Y_val).to(device)

    X_test_t  = torch.from_numpy(X_test).to(device)
    Y_test_t  = torch.from_numpy(Y_test).to(device)

    # ---- init a fresh model per fold (WFV requirement) ----
    model = Model(num_features=num_features, seq_len=seq_len).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # trackers (per fold)
    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = None

    # ---- TRAIN EPOCH LOOP (your loop) ----
    for epoch in range(num_epochs):
        dt0_epoch = datetime.now()

        mini_batches, seed = create_minibatches(X_train_t, Y_train_t, batch_size, seed, shuffle=True)

        train_loss = train_one_epoch(model, optimizer, criterion, mini_batches)

        # evaluate on VAL (NOT on test) during training
        val_loss, pred_val = eval_model(model, criterion, X_val_t, Y_val_t)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # keep best model by VAL loss (or VAL equity if you prefer)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"[Fold {fold}] Epoch {epoch:03d} | "
            f"TrainLoss {train_loss:.6g} | ValLoss {val_loss:.6g} | "
            f"Duration {datetime.now() - dt0_epoch}"
        )

    # ---- restore best model (by VAL) ----
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    # ---- choose threshold on VAL predictions only ----
    pred_val_np = pred_val.detach().cpu().numpy()    # last val preds (or recompute)
    thr = choose_threshold_from_val(pred_val_np, q=0.90)

    #Save Model:
    fold_ckpt_path = f"models/wfv_fold_{fold}_best.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "num_features": num_features,
        "seq_len": seq_len,
        "horizon": horizon,
        "thr": thr,   # will set after computing thr; see note below
    }, fold_ckpt_path)
    print(f"[Fold {fold}] Saved best model to {fold_ckpt_path}")

    # ---- final evaluation on TEST (OOS for this fold) ----
    test_loss, pred_test = eval_model(model, criterion, X_test_t, Y_test_t)
    pred_test_np = pred_test.detach().cpu().numpy()

    signals = predictions_to_signals(pred_test_np, thr)

    # IMPORTANT: no-overlap backtest for horizon strategy
    equity, equities, accuracy, traded = backtest_signals(
        opens_test,
        closes_test,
        signals,
        pos_size=1000.0,
        fee_rate=fee_rate
    )

    wf_results.append({
        "fold": fold,
        "start_idx": start,
        "thr": thr,
        "test_loss": test_loss,
        "equity": equity,
        "accuracy": accuracy,
        "traded": traded,
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
    })

    print(
        f"\n=== WFV Fold {fold} RESULT ===\n"
        f"Thr {thr:.6g} | TestLoss {test_loss:.6g} | "
        f"Equity {equity:,.2f} | Acc {accuracy:.2f}% | Traded {traded}\n"
    )

    # move forward
    start += STEP
    fold += 1


Streaming output truncated to the last 5000 lines.
[Fold 1] Epoch 062 | TrainLoss 0.000127459 | ValLoss 6.85122e-05 | Duration 0:00:01.118527
Random Seed: 166
[Fold 1] Epoch 063 | TrainLoss 0.000127282 | ValLoss 6.85096e-05 | Duration 0:00:01.112795
Random Seed: 166
[Fold 1] Epoch 064 | TrainLoss 0.000127105 | ValLoss 6.85069e-05 | Duration 0:00:01.118795
Random Seed: 166
[Fold 1] Epoch 065 | TrainLoss 0.000126929 | ValLoss 6.85041e-05 | Duration 0:00:01.124083
Random Seed: 166
[Fold 1] Epoch 066 | TrainLoss 0.000126753 | ValLoss 6.85013e-05 | Duration 0:00:01.271177
Random Seed: 166
[Fold 1] Epoch 067 | TrainLoss 0.000126577 | ValLoss 6.84984e-05 | Duration 0:00:01.404072
Random Seed: 166
[Fold 1] Epoch 068 | TrainLoss 0.000126402 | ValLoss 6.84954e-05 | Duration 0:00:01.105872
Random Seed: 166
[Fold 1] Epoch 069 | TrainLoss 0.000126226 | ValLoss 6.84925e-05 | Duration 0:00:01.125514
Random Seed: 166
[Fold 1] Epoch 070 | TrainLoss 0.000126051 | ValLoss 6.84895e-05 | Duration 0:00:01.1

In [33]:
eqs = [r["equity"] for r in wf_results]
print("Folds:", len(wf_results))
print("Mean final equity:", np.mean(eqs))
print("Median final equity:", np.median(eqs))
print("Profitable folds:", sum(e > 1000 for e in eqs), "/", len(eqs))


Folds: 26
Mean final equity: 600.2328785898907
Median final equity: 729.9482910488202
Profitable folds: 3 / 26
